# N-BEATS — Walmart Store Sales Forecasting

Model experiment notebook for the N-BEATS branch of the multi-model comparison
(LightGBM, XGBoost, N-BEATS, classical stats). N-BEATS is trained **globally**
(one shared-weight model across every Store-Dept series, not one model per
series) and is a **pure univariate** architecture per the original paper — no
exogenous features — so it does not reuse `utils/feature_engineering.py`.
Architecture, windowing, training-loop and ensembling code live in
`utils/nbeats_model.py`, `utils/nbeats_data.py`, `utils/nbeats_train.py` and
`utils/nbeats_ensemble.py`, shared with the standalone HPO script
`run_nbeats_hpo.py`. `utils/metrics.wmae` (the same function LightGBM uses) is
reused unchanged for every reported metric.

## Table of Contents
1. [Setup](#1)
2. [Local train/test split](#2)
3. [Walk-forward CV](#3)
4. [Global panel & windowing](#4)
5. [Baseline sanity check](#5)
6. [Hyperparameter search (Optuna)](#6)
7. [Final models & ensemble results](#7)
8. [Production model](#8)</cell id="ffb5f5b7">

<a id='1'></a>
## 1. Setup

In [1]:
import warnings
import pandas as pd
import numpy as np
import torch

warnings.filterwarnings('ignore', category=DeprecationWarning)

from utils.nbeats_data import build_panel
from utils.nbeats_train import (
    make_cv_folds, local_split_idx, max_windows_for_lookback, run_cv_for_config,
    HORIZON, LOCAL_TEST_WEEKS, INITIAL_TRAIN_WEEKS, VAL_WEEKS, N_FOLDS,
)
from utils.feature_engineering import HOLIDAY_DATES
from utils.metrics import wmae

pd.set_option('display.max_columns', 50)
torch.set_num_threads(16)

DATA_DIR = 'data/raw/walmart-recruiting-store-sales-forecasting/'
train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR + 'stores.csv')

print(f'train.csv: {train.shape}, {train.Date.min().date()} -> {train.Date.max().date()}')
print(f'test.csv : {test.shape}, {test.Date.min().date()} -> {test.Date.max().date()}')
print(f'n series (Store-Dept combos): {train.groupby(["Store","Dept"]).ngroups}')


train.csv: (421570, 5), 2010-02-05 -> 2012-10-26
test.csv : (115064, 4), 2012-11-02 -> 2013-07-26
n series (Store-Dept combos): 3331


<a id='2'></a>
## 2. Local Train/Test Split

Identical convention to `model_experiment_LightGBM.ipynb`: the last **52 weeks**
of `train.csv` are held out as a local test set (Kaggle's real `test.csv` has
no target, so model comparison during development needs a local proxy with
the same holiday composition — one Thanksgiving, one Christmas, one Super
Bowl). N-BEATS operates on a global (Store, Dept) panel rather than a merged
feature frame, so the split is expressed as **array indices** into that panel
instead of date filters — `utils.nbeats_train.local_split_idx` is the index
equivalent of the LightGBM notebook's `cutoff_date = unique_dates[-52]`, and
produces the exact same boundary (verified below).

In [2]:
sales, store_ids, dept_ids, store_types, all_dates, is_holiday = build_panel(train, stores)
n_series, n_dates = sales.shape
print(f'panel: {n_series} series x {n_dates} weeks ({all_dates.min().date()} -> {all_dates.max().date()})')

local_train_end_idx = local_split_idx(n_dates, LOCAL_TEST_WEEKS)
print(f'local_train: weeks 0..{local_train_end_idx} '
      f'({all_dates[0].date()} -> {all_dates[local_train_end_idx].date()}, {local_train_end_idx+1} wks)')
print(f'local_test : weeks {local_train_end_idx+1}..{n_dates-1} '
      f'({all_dates[local_train_end_idx+1].date()} -> {all_dates[-1].date()}, {n_dates-local_train_end_idx-1} wks)')

# Sanity-check against LightGBM's own date-based cutoff.
unique_dates = np.sort(train['Date'].unique())
lgbm_cutoff = unique_dates[-52]
assert all_dates[local_train_end_idx + 1] == pd.Timestamp(lgbm_cutoff), 'local-test boundary must match LightGBM exactly'
print('\nBoundary matches model_experiment_LightGBM.ipynb exactly: OK')


panel: 3331 series x 143 weeks (2010-02-05 -> 2012-10-26)
local_train: weeks 0..90 (2010-02-05 -> 2011-10-28, 91 wks)
local_test : weeks 91..142 (2011-11-04 -> 2012-10-26, 52 wks)

Boundary matches model_experiment_LightGBM.ipynb exactly: OK


In [3]:
def holidays_in_range(dates):
    dates_set = set(pd.to_datetime(dates))
    present = {}
    for name, hdates in HOLIDAY_DATES.items():
        matched = [d.date() for d in pd.to_datetime(hdates) if d in dates_set]
        if matched:
            present[name] = matched
    return present

print('local_train holidays:', holidays_in_range(all_dates[:local_train_end_idx+1]))
print('local_test  holidays:', holidays_in_range(all_dates[local_train_end_idx+1:]))


local_train holidays: {'SuperBowl': [datetime.date(2010, 2, 12), datetime.date(2011, 2, 11)], 'LaborDay': [datetime.date(2010, 9, 10), datetime.date(2011, 9, 9)], 'Thanksgiving': [datetime.date(2010, 11, 26)], 'Christmas': [datetime.date(2010, 12, 31)]}
local_test  holidays: {'SuperBowl': [datetime.date(2012, 2, 10)], 'LaborDay': [datetime.date(2012, 9, 7)], 'Thanksgiving': [datetime.date(2011, 11, 25)], 'Christmas': [datetime.date(2011, 12, 30)]}


<a id='3'></a>
## 3. Walk-Forward Cross-Validation

**Same fold boundaries as `model_experiment_LightGBM.ipynb`, by construction**:
`INITIAL_TRAIN_WEEKS=52`, `VAL_WEEKS=13`, `N_FOLDS=3`, expanding window, no
shuffling, folds live entirely inside `local_train` (Thanksgiving/Christmas
stay reserved for the final local-test evaluation, never seen during CV/HPO).

In [4]:
cv_folds = make_cv_folds(n_dates, INITIAL_TRAIN_WEEKS, VAL_WEEKS, N_FOLDS)
for i, (train_end_idx, val_start_idx, val_end_idx) in enumerate(cv_folds, start=1):
    print(f'Fold {i}: train {all_dates[0].date()} -> {all_dates[train_end_idx].date()} '
          f'({train_end_idx+1} wks) | val {all_dates[val_start_idx].date()} -> {all_dates[val_end_idx].date()} '
          f'({val_end_idx-val_start_idx+1} wks) | holidays: {holidays_in_range(all_dates[val_start_idx:val_end_idx+1])}')


Fold 1: train 2010-02-05 -> 2011-01-28 (52 wks) | val 2011-02-04 -> 2011-04-29 (13 wks) | holidays: {'SuperBowl': [datetime.date(2011, 2, 11)]}
Fold 2: train 2010-02-05 -> 2011-04-29 (65 wks) | val 2011-05-06 -> 2011-07-29 (13 wks) | holidays: {}
Fold 3: train 2010-02-05 -> 2011-07-29 (78 wks) | val 2011-08-05 -> 2011-10-28 (13 wks) | holidays: {'LaborDay': [datetime.date(2011, 9, 9)]}


<a id='4'></a>
## 4. Global Panel & Windowing

N-BEATS trains on fixed-length `(lookback, horizon)` windows of raw
`Weekly_Sales`, sliding across **every** Store-Dept series (`utils.nbeats_data
.build_training_windows`), with `horizon = VAL_WEEKS = 13` so a fold's
validation length *is* the forecast horizon. Per-window mean-scale
normalization (`utils.nbeats_data.scale_windows`) keeps the network's input
distribution sane across wildly different Store/Dept sales magnitudes; loss
is always computed after re-scaling forecasts back to real dollars, so
reported WMAE is in the same units as the competition metric.

**Lookback-multiple feasibility.** The task spec asks for an ensemble over
2x-7x the horizon. A `(fold, lookback)` combination only has trainable windows
if `fold_train_weeks >= lookback + horizon` (a window's own forecast portion
must still fit inside that fold's train range — the fold's *validation* range
is reserved for the held-out evaluation, never for gradient updates). Because
the walk-forward folds above are only 52/65/78 weeks (mirroring LightGBM
exactly), larger multiples run out of room fast:

In [5]:
rows = []
for mult in range(2, 8):
    lookback = mult * HORIZON
    row = {'multiplier': f'{mult}x', 'lookback_weeks': lookback}
    for i, (train_end_idx, _, _) in enumerate(cv_folds, start=1):
        row[f'fold{i}_windows'] = max(0, max_windows_for_lookback(train_end_idx + 1, lookback))
    row['local_train_windows'] = max(0, max_windows_for_lookback(local_train_end_idx + 1, lookback))
    rows.append(row)
feasibility_df = pd.DataFrame(rows)
feasibility_df


,multiplier,lookback_weeks,fold1_windows,fold2_windows,fold3_windows,local_train_windows
0,2x,26,14,27,40,53
1,3x,39,1,14,27,40
2,4x,52,0,1,14,27
3,5x,65,0,0,1,14
4,6x,78,0,0,0,1
5,7x,91,0,0,0,0


**Reading this table:** multiples 2x-3x have healthy window counts in every
fold and are safe to score with the 3-fold CV harness as-is. 4x-6x become
fold-limited (fold 1 in particular runs out of room) and 7x has **zero**
trainable windows even across the full 91-week `local_train` range — it needs
>= 104 training weeks, which only the full `train.csv` history (143 weeks)
provides.

This is why hyperparameter scoring (Section 6, and `run_nbeats_hpo.py`) uses
`utils.nbeats_train.run_cv_for_config`, which drops any fold that can't
produce at least `MIN_WINDOWS_PER_FOLD` windows for a given trial's lookback
and requires >= 2 valid folds to score a trial at all — configs that don't
fit the CV harness are pruned rather than silently scored on too little data.
The full 2x-7x ensemble (Section 7, next session) is trained in two tiers
instead: an *evaluable* tier (2x-6x, fit on `local_train` only, scored against
the real `local_test` holdout) and a *production* tier (2x-7x, fit on the full
143-week history) used for actual Kaggle `test.csv` inference — see
`utils/nbeats_ensemble.py` docstring for the full rationale.

<a id='5'></a>
## 5. Baseline Sanity Check

A small untuned generic-architecture N-BEATS (2 stacks x 2 blocks, 64-wide FC
layers, `lookback_multiplier=2`) run through the exact walk-forward CV harness
that `run_nbeats_hpo.py` uses for every Optuna trial — confirms the mechanism
itself (windowing, scaling, doubly-residual forward pass, WMAE-weighted loss,
early stopping) works end-to-end and produces a sane, stable WMAE before the
real hyperparameter search is trusted.

In [6]:
baseline_config = dict(
    architecture='generic', n_stacks=2, n_blocks=2, layer_size=64, n_fc_layers=2,
    lookback_multiplier=2, loss='wmae', batch_size=512, optimizer='adam',
    learning_rate=1e-3, weight_decay=0.0, dropout=0.0, share_weights=False,
)

baseline_result = run_cv_for_config(
    baseline_config, sales, is_holiday, cv_folds, max_epochs=15, patience=4, seed=42, verbose=False,
)

for i, fd in enumerate(baseline_result['fold_details'], start=1):
    print(f"Fold {i}: WMAE = {fd['wmae']:.2f}  (train windows={fd['n_train_windows']}, "
          f"epochs run={len(fd['history']['train_loss'])})")

print(f"\nMean WMAE across {baseline_result['n_folds_used']} folds: {baseline_result['mean_wmae']:.2f}")
print(f"Std WMAE: {baseline_result['std_wmae']:.2f}")


Fold 1: WMAE = 2504.74  (train windows=38383, epochs run=7)
Fold 2: WMAE = 2699.49  (train windows=73953, epochs run=5)
Fold 3: WMAE = 1997.41  (train windows=109574, epochs run=13)

Mean WMAE across 3 folds: 2400.55
Std WMAE: 295.94


<a id='6'></a>
## 6. Hyperparameter Search (Optuna)

Full search (architecture, stack/block counts, FC depth/width, lookback
multiplier, learning rate, batch size, optimizer, weight decay, loss function,
dropout, weight sharing) runs in the standalone script `run_nbeats_hpo.py`,
not inline in this notebook, so it can execute as a long/overnight job
independent of the notebook/kernel:

```
.venv/Scripts/python run_nbeats_hpo.py --n-trials 80 --max-epochs 40 --patience 6 \
    --study-name nbeats_hpo --storage sqlite:///nbeats_optuna.db
```

Every trial is scored with `run_cv_for_config` (Section 5's harness) and
logged as a **nested MLflow run** under experiment `NBEATS_Training`
(DagsHub-hosted), parent run `NBEATS_HPO_Search`. Trials whose sampled
`lookback_multiplier` doesn't have >= 2 valid CV folds (Section 4) are pruned.

**Status: paused at 30/80 requested trials** (resumable any time from the
same `nbeats_optuna.db` study — nothing below depends on finishing the rest
of the search). Paused specifically because running it concurrently with
`run_nbeats_finalize.py` caused severe CPU thread-oversubscription on this
16-core machine (each process's PyTorch ops independently grab up to 16
threads), so the two competed rather than sharing cleanly.</cell id="e13c243f">

In [7]:
import optuna

study = optuna.load_study(study_name='nbeats_hpo', storage='sqlite:///nbeats_optuna.db')
complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f'Trials attempted: {len(study.trials)} / 80 requested '
      f'({len(complete)} completed, {len(pruned)} pruned, '
      f'{len(study.trials) - len(complete) - len(pruned)} interrupted by the pause)')


def best_trial_for(archs):
    candidates = [t for t in complete if t.params.get('architecture') in archs]
    return min(candidates, key=lambda t: t.value) if candidates else None


best_generic = best_trial_for(['generic'])
best_interpretable = best_trial_for(['interpretable'])

print(f'\nBest generic (trial {best_generic.number}): CV mean WMAE = {best_generic.value:.2f}')
print(best_generic.params)
print(f'\nBest interpretable (trial {best_interpretable.number}): CV mean WMAE = {best_interpretable.value:.2f}')
print(best_interpretable.params)


Trials attempted: 30 / 80 requested (17 completed, 10 pruned, 3 interrupted by the pause)

Best generic (trial 23): CV mean WMAE = 1911.10
{'architecture': 'generic', 'n_stacks': 2, 'n_blocks': 2, 'n_fc_layers': 4, 'layer_size': 256, 'lookback_multiplier': 4, 'learning_rate': 0.0007279854007274201, 'batch_size': 256, 'optimizer': 'adam', 'weight_decay': 3.822799078402829e-06, 'loss': 'smape', 'dropout': 0.19094568604860213, 'share_weights': False}

Best interpretable (trial 4): CV mean WMAE = 1902.38
{'architecture': 'interpretable', 'n_blocks': 2, 'n_fc_layers': 5, 'layer_size': 512, 'lookback_multiplier': 4, 'learning_rate': 0.0010157001316706408, 'batch_size': 1024, 'optimizer': 'adam', 'weight_decay': 5.973412665154223e-06, 'loss': 'mae', 'dropout': 0.18797923060380148, 'share_weights': True, 'trend_degree': 4}


**Optuna diagnostics** (hyperparameter importance, parallel-coordinate view, optimization history over the 30 attempted trials):

![Optuna parameter importance](plots/nbeats_optuna_importance.png)
![Optuna parallel coordinate](plots/nbeats_optuna_parallel.png)
![Optuna optimization history](plots/nbeats_optuna_history.png)

<a id='7'></a>
## 7. Final Models & Ensemble Results

Retraining and evaluation runs in `run_nbeats_finalize.py` (a GPU-portable
version, `nbeats_colab_finalize.py`, was used to actually produce the numbers
below, on a Colab T4 GPU — the local CPU run hit the same thread-contention
problem as Section 6 when run alongside the HPO search). For each of the best
generic and best interpretable configs from Section 6, it:

1. Re-runs a single-fold CV pass to get a real loss curve and a fixed epoch
   count (mirrors `model_experiment_LightGBM.ipynb`'s `best_n_estimators`
   convention — see the Engineering Notes below for why).
2. Retrains a final model on the full 91-week `local_train` pool for that
   epoch count (no held-out validation slice wasted).
3. Evaluates it via 4 non-overlapping 13-week rolling-block forecasts across
   the full 52-week `local_test` holdout (re-anchored on true observed
   history each block, like re-forecasting weekly in production).

It then builds two ensemble tiers per the Section 4 feasibility split: an
**evaluable** tier (2x-6x H, 10 members = 2 base configs x 5 multipliers,
fit on `local_train` only, scored the same way) and a **production** tier
(2x-7x H, 6 members, fit on the *full* 143-week history — the only tier
where 7x H has enough training weeks to be trainable at all, and the actual
deliverable for scoring Kaggle's unlabeled `test.csv`).

**The cells below are the actual code and captured output from the Colab T4 GPU run** (`nbeats_colab_finalize.py`, executed as `Untitled1.ipynb`) that produced these results — not re-executed in this notebook's local CPU kernel. A CUDA device-placement bug was hit and fixed live during this run (monkey-patched in-session, then fixed properly in `utils/nbeats_plots.py` source — see Engineering Notes at the end of this notebook); the patch cells themselves are omitted here since the permanent source fix supersedes them.</cell id="f2d2de1b">

In [ ]:
BEST_GENERIC_CFG = dict(
    architecture='generic', n_stacks=2, n_blocks=2, layer_size=256, n_fc_layers=4,
    lookback_multiplier=4, loss='smape', batch_size=256, optimizer='adam',
    learning_rate=0.0007279854007274201, weight_decay=3.822799078402829e-06,
    dropout=0.19094568604860213, share_weights=False, trend_degree=3,
)
BEST_GENERIC_TRIAL_NUM = 23
BEST_GENERIC_CV_WMAE = 1911.095445291105

BEST_INTERPRETABLE_CFG = dict(
    architecture='interpretable', n_blocks=2, layer_size=512, n_fc_layers=5,
    lookback_multiplier=4, loss='mae', batch_size=1024, optimizer='adam',
    learning_rate=0.0010157001316706408, weight_decay=5.973412665154223e-06,
    dropout=0.18797923060380148, share_weights=True, trend_degree=4,
)
BEST_INTERPRETABLE_TRIAL_NUM = 4
BEST_INTERPRETABLE_CV_WMAE = 1902.3813188070403

In [ ]:
train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR + 'stores.csv')
sales, store_ids, dept_ids, store_types, all_dates, is_holiday = build_panel(train, stores)
n_dates = sales.shape[1]
local_train_end_idx = local_split_idx(n_dates)
cv_folds = make_cv_folds(n_dates)


def series_label(s):
    return f'Store {store_ids[s]} Dept {dept_ids[s]}'


def per_series_wmae(y_true, y_pred, holiday_grid, series_idx):
    abs_err = np.abs(y_true - y_pred)
    weights = np.where(holiday_grid, 5, 1)
    out = {}
    for s in np.unique(series_idx):
        mask = series_idx == s
        w = weights[mask]
        out[s] = float((abs_err[mask] * w).sum() / w.sum())
    return out


def breakdown_wmae(y_true, y_pred, holiday_grid, series_idx, store_types):
    holiday_mask = holiday_grid.astype(bool)
    overall = compute_wmae(y_true, y_pred, holiday_grid)
    holiday_wmae = compute_wmae(y_true[holiday_mask], y_pred[holiday_mask], holiday_grid[holiday_mask]) \
        if holiday_mask.any() else float('nan')
    non_holiday_wmae = compute_wmae(y_true[~holiday_mask], y_pred[~holiday_mask], holiday_grid[~holiday_mask])
    type_wmae = {}
    for t in ['A', 'B', 'C']:
        type_series = set(np.where(store_types == t)[0])
        mask = np.array([s in type_series for s in series_idx])
        if mask.sum() == 0:
            continue
        type_wmae[t] = compute_wmae(y_true[mask], y_pred[mask], holiday_grid[mask])
    return overall, holiday_wmae, non_holiday_wmae, type_wmae


def loss_curve_and_epochs(cfg, max_epochs=60, patience=8):
    result = run_cv_for_config(cfg, sales, is_holiday, cv_folds[-1:], max_epochs=max_epochs,
                                patience=patience, min_folds=1, device=DEVICE)
    if result is None:
        return {'train_loss': [], 'val_wmae': []}, max_epochs // 2
    fd = result['fold_details'][0]
    return fd['history'], max(1, int(round(fd['best_epoch'])))


results = {}

In [ ]:
cfg = dict(BEST_GENERIC_CFG)
print(f'Best generic config (trial {BEST_GENERIC_TRIAL_NUM}, CV mean_wmae={BEST_GENERIC_CV_WMAE:.2f}): {cfg}')
with mlflow.start_run(run_name='NBEATS_Best_Generic_Colab'):
    mlflow.log_params(cfg)
    mlflow.log_metric('cv_mean_wmae', BEST_GENERIC_CV_WMAE)
    mlflow.log_param('device', DEVICE)

    curve_hist, n_epochs = loss_curve_and_epochs(cfg)
    mlflow.log_metric('final_fit_n_epochs', n_epochs)
    for ep, (tl, vw) in enumerate(zip(curve_hist['train_loss'], curve_hist['val_wmae']), start=1):
        mlflow.log_metric('train_loss', tl, step=ep)
        mlflow.log_metric('val_wmae', vw, step=ep)

    model_g, hist_g, lb_g = train_final_member(cfg, sales, is_holiday, local_train_end_idx, n_epochs, device=DEVICE)

    y_true, y_pred, hgrid, sidx, block, step = rolling_block_forecast(
        model_g, lb_g, sales, is_holiday, local_train_end_idx, HORIZON, n_blocks=4)
    overall, hol, nonhol, by_type = breakdown_wmae(y_true, y_pred, hgrid, sidx, store_types)
    mlflow.log_metric('local_test_wmae_overall', overall)
    mlflow.log_metric('local_test_wmae_holiday', hol)
    mlflow.log_metric('local_test_wmae_non_holiday', nonhol)
    for t, v in by_type.items():
        mlflow.log_metric(f'local_test_wmae_type_{t}', v)

    P.plot_loss_curves(curve_hist, 'Best generic (Colab GPU) -- train/val curves', f'{PLOTS_DIR}/nbeats_loss_curves_generic_colab.png')
    mlflow.log_artifact(f'{PLOTS_DIR}/nbeats_loss_curves_generic_colab.png')
    torch.save(model_g.state_dict(), f'{MODELS_DIR}/nbeats_best_generic_colab.pt')
    mlflow.log_artifact(f'{MODELS_DIR}/nbeats_best_generic_colab.pt')

    print(f'  local_test WMAE: overall={overall:.2f} holiday={hol:.2f} non_holiday={nonhol:.2f} by_type={by_type}')
    results['best_generic'] = dict(
        trial=BEST_GENERIC_TRIAL_NUM, config=cfg, cv_mean_wmae=BEST_GENERIC_CV_WMAE,
        local_test_wmae_overall=overall, local_test_wmae_holiday=hol,
        local_test_wmae_non_holiday=nonhol, local_test_wmae_by_type=by_type,
    )


Best generic config (trial 23, CV mean_wmae=1911.10): {'architecture': 'generic', 'n_stacks': 2, 'n_blocks': 2, 'layer_size': 256, 'n_fc_layers': 4, 'lookback_multiplier': 4, 'loss': 'smape', 'batch_size': 256, 'optimizer': 'adam', 'learning_rate': 0.0007279854007274201, 'weight_decay': 3.822799078402829e-06, 'dropout': 0.19094568604860213, 'share_weights': False, 'trend_degree': 3}
  local_test WMAE: overall=2161.25 holiday=3134.43 non_holiday=1756.13 by_type={'A': np.float64(2495.684571692365), 'B': np.float64(2003.3678912107828), 'C': np.float64(910.3641942846579)}
🏃 View run NBEATS_Best_Generic_Colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/e9f00aa1857a4ea493148e53b8766412
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1


In [ ]:
cfg = dict(BEST_INTERPRETABLE_CFG)
print(f'Best interpretable config (trial {BEST_INTERPRETABLE_TRIAL_NUM}, CV mean_wmae={BEST_INTERPRETABLE_CV_WMAE:.2f}): {cfg}')
with mlflow.start_run(run_name='NBEATS_Best_Interpretable_Colab'):
    mlflow.log_params(cfg)
    mlflow.log_metric('cv_mean_wmae', BEST_INTERPRETABLE_CV_WMAE)
    mlflow.log_param('device', DEVICE)

    curve_hist_i, n_epochs_i = loss_curve_and_epochs(cfg)
    mlflow.log_metric('final_fit_n_epochs', n_epochs_i)
    for ep, (tl, vw) in enumerate(zip(curve_hist_i['train_loss'], curve_hist_i['val_wmae']), start=1):
        mlflow.log_metric('train_loss', tl, step=ep)
        mlflow.log_metric('val_wmae', vw, step=ep)

    model_i, hist_i, lb_i = train_final_member(cfg, sales, is_holiday, local_train_end_idx, n_epochs_i, device=DEVICE)

    y_true, y_pred, hgrid, sidx, block, step = rolling_block_forecast(
        model_i, lb_i, sales, is_holiday, local_train_end_idx, HORIZON, n_blocks=4)
    overall, hol, nonhol, by_type = breakdown_wmae(y_true, y_pred, hgrid, sidx, store_types)
    mlflow.log_metric('local_test_wmae_overall', overall)
    mlflow.log_metric('local_test_wmae_holiday', hol)
    mlflow.log_metric('local_test_wmae_non_holiday', nonhol)
    for t, v in by_type.items():
        mlflow.log_metric(f'local_test_wmae_type_{t}', v)

    P.plot_loss_curves(curve_hist_i, 'Best interpretable (Colab GPU) -- train/val curves', f'{PLOTS_DIR}/nbeats_loss_curves_interpretable_colab.png')
    mlflow.log_artifact(f'{PLOTS_DIR}/nbeats_loss_curves_interpretable_colab.png')

    Xe, series_idx_i = build_eval_window(sales, lb_i, local_train_end_idx)
    P.plot_decomposition(model_i, Xe[0], all_dates, series_label(series_idx_i[0]), f'{PLOTS_DIR}/nbeats_decomposition_colab.png')
    P.plot_backcast_reconstruction(model_i, Xe[0], series_label(series_idx_i[0]), f'{PLOTS_DIR}/nbeats_backcast_reconstruction_colab.png')
    mlflow.log_artifact(f'{PLOTS_DIR}/nbeats_decomposition_colab.png')
    mlflow.log_artifact(f'{PLOTS_DIR}/nbeats_backcast_reconstruction_colab.png')

    torch.save(model_i.state_dict(), f'{MODELS_DIR}/nbeats_best_interpretable_colab.pt')
    mlflow.log_artifact(f'{MODELS_DIR}/nbeats_best_interpretable_colab.pt')

    print(f'  local_test WMAE: overall={overall:.2f} holiday={hol:.2f} non_holiday={nonhol:.2f} by_type={by_type}')
    results['best_interpretable'] = dict(
        trial=BEST_INTERPRETABLE_TRIAL_NUM, config=cfg, cv_mean_wmae=BEST_INTERPRETABLE_CV_WMAE,
        local_test_wmae_overall=overall, local_test_wmae_holiday=hol,
        local_test_wmae_non_holiday=nonhol, local_test_wmae_by_type=by_type,
    )

Best interpretable config (trial 4, CV mean_wmae=1902.38): {'architecture': 'interpretable', 'n_blocks': 2, 'layer_size': 512, 'n_fc_layers': 5, 'lookback_multiplier': 4, 'loss': 'mae', 'batch_size': 1024, 'optimizer': 'adam', 'learning_rate': 0.0010157001316706408, 'weight_decay': 5.973412665154223e-06, 'dropout': 0.18797923060380148, 'share_weights': True, 'trend_degree': 4}
  local_test WMAE: overall=2286.51 holiday=3352.35 non_holiday=1842.81 by_type={'A': np.float64(2646.1527168860102), 'B': np.float64(2113.3346235037384), 'C': np.float64(955.6731296576778)}
🏃 View run NBEATS_Best_Interpretable_Colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/30b633ead48e454fb5f242e0667dae13
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1


In [ ]:
base_configs = [BEST_GENERIC_CFG, BEST_INTERPRETABLE_CFG]
print(f'Training evaluable ensemble: multipliers {EVALUABLE_MULTIPLIERS} x {len(base_configs)} base config(s)')
with mlflow.start_run(run_name='NBEATS_Ensemble_Evaluable_Colab'):
    mlflow.log_param('multipliers', EVALUABLE_MULTIPLIERS)
    mlflow.log_param('n_base_configs', len(base_configs))
    mlflow.log_param('device', DEVICE)
    members = []
    for base_cfg in base_configs:
        _, base_n_epochs = loss_curve_and_epochs(base_cfg)
        for mult in EVALUABLE_MULTIPLIERS:
            cfg = dict(base_cfg, lookback_multiplier=mult)
            with mlflow.start_run(run_name=f'member_{base_cfg["architecture"]}_{mult}x_colab', nested=True):
                mlflow.log_params(cfg)
                mlflow.log_metric('n_epochs', base_n_epochs)
                model_m, hist_m, lb_m = train_final_member(
                    cfg, sales, is_holiday, local_train_end_idx, base_n_epochs, device=DEVICE)
                print(f'  member {base_cfg["architecture"]} x{mult}: trained {len(hist_m["train_loss"])} epochs, '
                      f'final train_loss={hist_m["train_loss"][-1]:.2f}')
                members.append({'model': model_m, 'lookback': lb_m, 'config': cfg})

    y_true, y_pred, hgrid, sidx, block, step = ensemble_rolling_forecast(
        members, sales, is_holiday, local_train_end_idx, HORIZON, n_blocks=4)
    overall, hol, nonhol, by_type = breakdown_wmae(y_true, y_pred, hgrid, sidx, store_types)
    mlflow.log_metric('local_test_wmae_overall', overall)
    mlflow.log_metric('local_test_wmae_holiday', hol)
    mlflow.log_metric('local_test_wmae_non_holiday', nonhol)
    for t, v in by_type.items():
        mlflow.log_metric(f'local_test_wmae_type_{t}', v)
    print(f'  ENSEMBLE local_test WMAE: overall={overall:.2f} holiday={hol:.2f} non_holiday={nonhol:.2f} by_type={by_type}')

    abs_err = np.abs(y_true - y_pred)
    wmae_by_series = per_series_wmae(y_true, y_pred, hgrid, sidx)
    series_wmae_arr = np.array(list(wmae_by_series.values()))
    series_keys_arr = np.array(list(wmae_by_series.keys()))
    order = np.argsort(series_wmae_arr)
    best_s, med_s, worst_s = series_keys_arr[order[0]], series_keys_arr[order[len(order) // 2]], series_keys_arr[order[-1]]

    examples = []
    for label, s in [('Best', best_s), ('Median', med_s), ('Worst', worst_s)]:
        mask = sidx == s
        examples.append((f'{label}: {series_label(s)}', y_true[mask].flatten(), y_pred[mask].flatten(), wmae_by_series[s]))
    P.plot_forecast_vs_actual(examples, f'{PLOTS_DIR}/nbeats_forecast_vs_actual_colab.png')
    P.plot_error_by_horizon(step, abs_err, f'{PLOTS_DIR}/nbeats_error_by_horizon_colab.png')
    P.plot_wmae_distribution(series_wmae_arr, f'{PLOTS_DIR}/nbeats_wmae_distribution_colab.png')
    P.plot_wmae_breakdown(hol, nonhol, by_type, f'{PLOTS_DIR}/nbeats_wmae_breakdown_colab.png')

    row_dates = np.array([
        all_dates[local_train_end_idx + 1 + b * HORIZON: local_train_end_idx + 1 + b * HORIZON + HORIZON]
        for b in block
    ])
    resid = y_true - y_pred
    mean_resid_by_date = pd.Series(resid.flatten(), index=row_dates.flatten()).groupby(level=0).mean().sort_index()
    P.plot_residual_diagnostics(mean_resid_by_date.values, mean_resid_by_date.index, f'{PLOTS_DIR}/nbeats_residuals_colab')

    for fname in ['nbeats_forecast_vs_actual_colab.png', 'nbeats_error_by_horizon_colab.png',
                  'nbeats_wmae_distribution_colab.png', 'nbeats_wmae_breakdown_colab.png',
                  'nbeats_residuals_colab_timeseries.png', 'nbeats_residuals_colab_hist.png', 'nbeats_residuals_colab_acf.png']:
        mlflow.log_artifact(f'{PLOTS_DIR}/{fname}')

    results['ensemble_evaluable'] = dict(
        multipliers=EVALUABLE_MULTIPLIERS, n_members=len(members),
        local_test_wmae_overall=overall, local_test_wmae_holiday=hol,
        local_test_wmae_non_holiday=nonhol, local_test_wmae_by_type=by_type,
    )


Training evaluable ensemble: multipliers [2, 3, 4, 5, 6] x 2 base config(s)
  member generic x2: trained 19 epochs, final train_loss=0.21
🏃 View run member_generic_2x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/fb54022bcd47409c8da184df0a443221
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  member generic x3: trained 19 epochs, final train_loss=0.17
🏃 View run member_generic_3x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/8b4e8428727f46e7ace1e406925f36c0
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  member generic x4: trained 19 epochs, final train_loss=0.13
🏃 View run member_generic_4x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/283301979b5842c796d411f790b7ef1e
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  member generic x5: trained 19 e

In [ ]:
print(f'Training production ensemble: multipliers {PRODUCTION_MULTIPLIERS} (full history, no held-out eval)')
with mlflow.start_run(run_name='NBEATS_Production_Final_Colab'):
    mlflow.log_param('multipliers', PRODUCTION_MULTIPLIERS)
    mlflow.log_param('device', DEVICE)
    prod_cfg = base_configs[0]
    _, prod_n_epochs = loss_curve_and_epochs(prod_cfg)
    mlflow.log_metric('n_epochs', prod_n_epochs)
    for mult in PRODUCTION_MULTIPLIERS:
        cfg = dict(prod_cfg, lookback_multiplier=mult)
        with mlflow.start_run(run_name=f'prod_member_{mult}x_colab', nested=True):
            mlflow.log_params(cfg)
            model_p, hist_p, lb_p = train_final_member(
                cfg, sales, is_holiday, n_dates - 1, prod_n_epochs, device=DEVICE)
            torch.save(model_p.state_dict(), f'{MODELS_DIR}/nbeats_production_{mult}x_colab.pt')
            mlflow.log_artifact(f'{MODELS_DIR}/nbeats_production_{mult}x_colab.pt')
            print(f'  production member x{mult}: trained {len(hist_p["train_loss"])} epochs, '
                  f'final train_loss={hist_p["train_loss"][-1]:.2f}')
    results['production_ensemble'] = dict(multipliers=PRODUCTION_MULTIPLIERS, base_config=prod_cfg)

Training production ensemble: multipliers [2, 3, 4, 5, 6, 7] (full history, no held-out eval)
  production member x2: trained 19 epochs, final train_loss=0.20
🏃 View run prod_member_2x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/10313fbf72274c31bde66ce31a463cbf
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  production member x3: trained 19 epochs, final train_loss=0.18
🏃 View run prod_member_3x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/211518c4e6584777b10248de6f13ea37
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  production member x4: trained 19 epochs, final train_loss=0.14
🏃 View run prod_member_4x_colab at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1/runs/436dbf42776444d99323f7273ca66d74
🧪 View experiment at: https://dagshub.com/tgela23/ml-final-project.mlflow/#/experiments/1
  production me

In [ ]:
with open(f'{REPORTS_DIR}/nbeats_finalize_results_colab.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('Done. Results written to reports/nbeats_finalize_results_colab.json')

Done. Results written to reports/nbeats_finalize_results_colab.json


Aggregated into a single comparison table:

In [8]:
import json

with open('reports/nbeats_finalize_results.json') as f:
    finalize_results = json.load(f)

rows = []
for label, key in [('Best Generic (single)', 'best_generic'),
                    ('Best Interpretable (single)', 'best_interpretable'),
                    ('Evaluable Ensemble (10 members)', 'ensemble_evaluable')]:
    d = finalize_results[key]
    row = {'model': label, 'overall': d['local_test_wmae_overall'],
           'holiday': d['local_test_wmae_holiday'], 'non_holiday': d['local_test_wmae_non_holiday']}
    row.update({f'type_{t}': v for t, v in d['local_test_wmae_by_type'].items()})
    rows.append(row)

results_df = pd.DataFrame(rows).set_index('model').round(2)
results_df


,overall,holiday,non_holiday,type_A,type_B,type_C
model,,,,,,
Best Generic (single),2161.25,3134.43,1756.13,2495.68,2003.37,910.36
Best Interpretable (single),2286.51,3352.35,1842.81,2646.15,2113.33,955.67
Evaluable Ensemble (10 members),2331.71,3502.22,1843.94,2675.32,2198.68,908.22


**Note:** the single best-generic model (trial 23) beat both the single
best-interpretable model and the full 10-member ensemble on this holdout —
ensembling did not help here. With only a 52-week/4-block holdout this isn't
strong evidence that ensembling is bad in general, just that it didn't pay
off on *this* holdout.

**Loss curves** (train loss / validation WMAE per epoch, from the single-fold CV re-run used to pick each final model's epoch count):

![Generic loss curves](plots/nbeats_loss_curves_generic.png)
![Interpretable loss curves](plots/nbeats_loss_curves_interpretable.png)

**Interpretable decomposition** (N-BEATS' signature interpretability output — trend + seasonality components of one series' forecast) **and backcast reconstruction** (how well the model reconstructs its own lookback window):

![Decomposition](plots/nbeats_decomposition.png)
![Backcast reconstruction](plots/nbeats_backcast_reconstruction.png)

**Ensemble evaluation plots** (best/median/worst-WMAE series, error by forecast step, WMAE distribution across all series, and the holiday/Store-Type breakdown table above visualized):

![Forecast vs actual](plots/nbeats_forecast_vs_actual.png)
![Error by horizon step](plots/nbeats_error_by_horizon.png)
![WMAE distribution](plots/nbeats_wmae_distribution.png)
![WMAE breakdown](plots/nbeats_wmae_breakdown.png)

**Residual diagnostics** (mean residual over time, histogram, autocorrelation — checking for systematic bias or leftover structure the model missed):

![Residuals over time](plots/nbeats_residuals_timeseries.png)
![Residuals histogram](plots/nbeats_residuals_hist.png)
![Residuals ACF](plots/nbeats_residuals_acf.png)

<a id='8'></a>
## 8. Production Model

6 members (multipliers 2x-7x H), each retrained from the best-generic config
(trial 23) on the **full 143-week `train.csv` history** — this is the only
tier where the 7x H lookback has enough history to train at all (see Section
4's feasibility table: 7x needs >= 104 training weeks, `local_train` only has
91). This is the actual deliverable for scoring Kaggle's real `test.csv`; it
has no reportable WMAE of its own since that file has no labels — the
evaluable-tier numbers in Section 7 are the best available proxy for expected
performance. Trained weights are logged as MLflow artifacts on the nested
`prod_member_{2..7}x` runs (see below).

## MLflow runs

All at https://dagshub.com/tgela23/ml-final-project.mlflow, experiment `NBEATS_Training`:

- Parent HPO run: `NBEATS_HPO_Search` (nested run per trial, named by trial number).
- `NBEATS_Best_Generic_Colab`, `NBEATS_Best_Interpretable_Colab` — the single final fits reported in Section 7.
- `NBEATS_Ensemble_Evaluable_Colab` (nested `member_{arch}_{mult}x_colab` per member).
- `NBEATS_Production_Final_Colab` (nested `prod_member_{mult}x_colab` per member).

Full write-up: `reports/NBEATS_summary.md`. Raw numbers: `reports/nbeats_finalize_results.json`.

---
## Engineering notes

Two real bugs were found and fixed while building `run_nbeats_finalize.py`
after this notebook was first executed — worth knowing if extending this
further:

1. **Final-model training originally reserved a 13-week validation block for
   early stopping**, which silently made `lookback_multiplier=6` untrainable
   (0 windows) and `5` nearly degenerate (1 window) within `local_train`'s
   91-week pool — contradicting the feasibility table in Section 4. Fixed:
   final members now train for a **fixed epoch count derived from CV**
   (`utils.nbeats_train.train_fixed_epochs`, epoch count = a fresh
   single-fold CV re-run's best epoch), mirroring exactly how
   `model_experiment_LightGBM.ipynb` picks `best_n_estimators` from the mean
   best-iteration across CV folds, rather than wasting training data on a
   redundant validation slice.
2. `utils.nbeats_train.run_cv_for_config` requires >= 2 valid CV folds by
   default (returns `None` otherwise) — a single-fold diagnostic re-run
   (used to get a real loss-curve plot + epoch count for the final fit) was
   silently hitting this guard and always returning the empty fallback.
   Fixed with a `min_folds` parameter (`run_nbeats_finalize.py` passes
   `min_folds=1` for its single-fold re-runs).
3. **GPU support** (`utils/nbeats_train.py`, `utils/nbeats_ensemble.py`,
   `utils/nbeats_plots.py`) was added later to run the finalize stage on a
   Colab T4 after the local CPU run proved too slow when run alongside the
   HPO search — `build_model`/`run_cv_for_config`/`train_final_member` take
   an optional `device` kwarg (default `'cpu'`, so every existing call site
   is unaffected), and `forecast_series`/`plot_decomposition`/
   `plot_backcast_reconstruction` auto-detect the model's device rather than
   assuming CPU.

Both fixes are verified with smoke tests; Section 5's baseline cell above
predates both bugs' fixes but is unaffected by either (it calls
`run_cv_for_config` with all 3 real folds, never hitting either guard).</cell id="740b98b3">